In [1]:
!pip install numpy
!pip install torch

In [2]:
import numpy as np
import torch
import torch.nn as nn

In [3]:
class PINN(nn.Module):
    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [2, 50, 50, 50, 50, 1]
        self.actiation = nn.Tanh()
        self.layers = nn.ModuleList()

        for i in range(len(layers)-1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        for layer in self.layers[:-1]:
            inputs = self.activation(layer(inputs))

        return self.layers[-1](inputs)

In [12]:
def compute_wave_equation_residue(model, x, t, c):
    psi = model(x, t)
    psi_t = torch.autograd.grad(psi, t, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psi_t_t = torch.autograd.grad(psi_t, t, torch.ones_like(psi_t), create_graph=True, retain_graph=True)[0]
    psi_x = torch.autograd.grad(psi, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psi_x_x = torch.autograd.grad(psi_x, x, torch.ones_like(psi_x), create_graph=True, retain_graph=True)[0]

    residue = psi_x_x - 1 / (c*c) * psi_t_t
    return (residue ** 2).mean()

In [13]:
def analytical_solution(x, t, c, A=1., L=1.):
    return A * np.sin(np.pi*x/L) * np.cos(np.pi*x*t/L)